# 05 · Defenses and situations

1. **Defense scheme tendencies:** how much does a defense blitz and stack the box *relative to what the situation and the offense's formation call for*? Are those tendencies real, and do defenses form archetypes?
2. **Style × defense:** does the value of an opening style (Phase 4) depend on the defense it faces? (held-out test)
3. **Situations:** which call types work in the red zone, the two-minute drill and the fourth quarter, and how does pass depth fare against blitz and man/zone looks?

Scheme (how a defense lines up) and results (EPA allowed, pressure) are kept apart, as with offensive style vs outcome.
Defensive looks come from nflverse participation data: box counts and pass rushers for all seasons, man/zone only from 2018.

In [ ]:
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
from drivedna.models.cluster import REPORTS_DIR
from drivedna.analysis.openings import OPENING_ORDER

R = lambda name: pd.read_csv(REPORTS_DIR / name)
tend, rel, arch = R("defense_season_scheme.csv"), R("defense_reliability.csv"), R("defense_archetype_stability.csv")
tend["label"] = tend["defteam"] + " " + tend["season"].astype(str)
print(len(tend), "defense-seasons;", int(tend["n_plays"].mean()), "plays each on average")

## 1 · Defense scheme tendencies
Excess over expectation: box = average defenders in the box above what down, distance, field position, score, time, quarter, season and the offense's formation predict;
blitz = share of dropbacks with 5+ pass rushers above expectation (big blitz: 6+); man = share of man coverage (2018+).

In [ ]:
for col in ["blitz_excess", "box_excess"]:
    top = tend.sort_values(col, ascending=False).head(6)[["label", col]].assign(**{col: lambda d: (d[col] * 100).round(1)})
    bot = tend.sort_values(col).head(4)[["label", col]].assign(**{col: lambda d: (d[col] * 100).round(1)})
    print(f"{col} ({'percentage points of dropbacks' if col.startswith('blitz') else 'x100 defenders in box'}) highest:", top.values.tolist())
    print(f"{'':>12}lowest:", bot.values.tolist())

In [ ]:
print("Is it a real, persistent trait?"); print(rel.round(2).to_string(index=False))
fig = px.scatter(tend, x="blitz_excess", y="box_excess", color="archetype", hover_name="label", height=560,
                 title="Defense-seasons: blitz tendency vs. box tendency (relative to situation)")
fig.show()

### Do defenses form archetypes?
Same tests as for offensive styles: silhouette, refits on bootstrap resamples, and a model fit on 2016-2020 vs. one fit on 2021-2025.

In [ ]:
print(arch.round(2).to_string(index=False))
prof = tend.groupby("archetype")[["box_excess", "blitz_excess", "big_blitz_excess", "epa_pass_allowed", "epa_run_allowed", "pressure_rate", "sack_rate", "explosive_allowed"]].mean().round(3)
prof["share_of_defense_seasons"] = tend["archetype"].value_counts(normalize=True).round(2)
print(); print(prof.T.to_string())

In [ ]:
res = ["epa_pass_allowed", "epa_run_allowed", "pressure_rate", "sack_rate", "explosive_allowed"]
print("Correlation of scheme tendencies with results (descriptive; results never define a scheme):")
print(tend[["box_excess", "blitz_excess", "big_blitz_excess"] + res].corr().loc[["box_excess", "blitz_excess", "big_blitz_excess"], res].round(2).to_string())

## 2 · Does the value of an opening style depend on the defense?
Ridge model of points per drive with opening style × opponent scheme (leave-one-game-out blitz and box tendencies, standardized). Tested **out of sample, leaving each season out**:
if the interactions are real, they must improve held-out prediction.

In [ ]:
cv = R("interaction_cv.csv"); print(cv.round(4).to_string(index=False))
eff = R("interaction_effects.csv")
print("\nEffect of each opening style (points/drive vs. typical) with the opponent's tendency at -1 SD vs +1 SD:")
print(eff.round(3).to_string(index=False))
print(f"\nSignificant at 5%: {int(eff['significant'].sum())} of {len(eff)} (about {0.05 * len(eff):.1f} expected by chance)")

## 3 · Which calls work in which situations
For plays that used a call, its EPA minus the typical call mix's EPA in the same down, distance, field position, score and time (fit within each situation; 95% intervals resample games).
**EPA does not value the clock**, which matters for running in the fourth quarter with a lead, so read run numbers there as incomplete.

In [ ]:
mix = R("situation_call_mix.csv").set_index("Unnamed: 0"); print("Call mix by situation:"); print(mix.round(3).to_string())
for target in ["epa", "success"]:
    st = R(f"situation_effects_{target}.csv")
    st["txt"] = st.apply(lambda r: f"{r.effect:+.3f} [{r.lo:+.3f},{r.hi:+.3f}]", axis=1)
    print(f"\n== {target.upper()} vs. typical call (n plays: {st.drop_duplicates('situation').set_index('situation')['n_plays'].to_dict()})" if target == "epa" else f"\n== {target.upper()} vs. typical call")
    print(st.pivot(index="situation", columns="call", values="txt")[["R_in", "R_out", "P_short", "P_mid", "P_deep"]].to_string())

In [ ]:
st = R("situation_effects_epa.csv")
fig = px.bar(st, x="call", y="effect", color="situation", barmode="group", error_y=st["hi"] - st["effect"], error_y_minus=st["effect"] - st["lo"],
             category_orders={"call": ["R_in", "R_out", "P_short", "P_mid", "P_deep"]}, height=520, title="EPA per play vs. the typical call, by situation")
fig.show()

## 4 · Pass depth against the defensive look
Thrown passes only (a sack has no depth). Sacks per dropback are shown separately because blitzes produce more of them, which this comparison leaves out.

In [ ]:
d = R("depth_by_look.csv")
d["txt"] = d.apply(lambda r: f"{r.effect:+.3f} [{r.lo:+.3f},{r.hi:+.3f}]", axis=1)
for t, g in d.groupby("look_type"):
    print(f"== {t}   (EPA per attempt vs. the typical depth, same situation)")
    print(g.pivot(index="look", columns="call", values="txt")[["P_short", "P_mid", "P_deep"]].to_string())
    print("raw EPA/attempt:"); print(g.pivot(index="look", columns="call", values="raw_mean")[["P_short", "P_mid", "P_deep"]].round(3).to_string())
    print("share of attempts by depth:"); print(g.pivot(index="look", columns="call", values="share")[["P_short", "P_mid", "P_deep"]].round(3).to_string())
print(); print(R("sack_by_look.csv").round(3).to_string(index=False))

## Takeaways

**1. Defense scheme tendencies are real, and more reliable than offensive ones.** Split-half reliability is 0.83-0.88 and year-to-year correlation 0.45-0.53 for box, blitz, big-blitz and man-coverage tendencies. The extremes are recognizable: the most blitz-heavy seasons are Minnesota 2023 (+26.6 points of dropbacks above what the situation predicts) and 2025, Baltimore 2019 and the Giants 2023 (consistent with Brian Flores' and Wink Martindale's well-known schemes), and the least are the Chargers in 2019-20 (about -15).

**2. Defenses split into two archetypes at most.** A two-way split is solid (k=2: bootstrap ARI 0.94, early-vs-late-seasons ARI 0.90, silhouette 0.40, the best separation anywhere in this project): *Aggressive* (37% of defense-seasons: box +0.11 defenders, blitz +6.6 points) vs *Light* (63%). From k=3 the partitions fall apart (bootstrap ARI ≤ 0.70, early-vs-late ≤ 0.63), so there is no richer taxonomy of defenses. The two groups differ only slightly on results: EPA allowed per dropback 0.033 vs 0.055, pressure rate 29.4% vs 27.2%.

**3. The value of an opening style does not detectably depend on the defense.** Held out by season, adding opening style reduces the error in predicting points per drive by 0.029 (CI 0.020 to 0.038), but adding style x opponent blitz/box tendency does not help (change -0.0017, CI -0.0037 to +0.0003; slightly worse). None of the 10 individual effect shifts (+1 SD vs -1 SD of opponent tendency) is significant, which is fewer than the 0.5 expected by chance. So the Phase 4 style results can be read as roughly defense-independent along these two dimensions.

**4. Which calls work.** Mid-depth passes are the reliable standout (+0.24 EPA vs the typical call and +16 points of success rate). Deep passes have the highest EPA (+0.41) but are boom-or-bust: success rate is only +1 point overall and *negative* in the red zone (-4.5) and two-minute drill (-5.7), though EPA stays positive everywhere (+0.27 to +0.48). Short passes are slightly below typical (-0.07) and runs well below (inside -0.20, outside -0.12). In the red zone the gaps narrow (inside runs -0.11) and deep balls are rare (4.6% of calls). Short passes are notably worse when trailing in the fourth quarter (-0.12 EPA) than when leading (-0.02).

**5. Against the blitz and man coverage, the deep-pass edge shrinks.** Deep-pass EPA per attempt vs the typical depth is +0.20 against a blitz (5+ rushers) vs +0.33 without, and +0.19 against man vs +0.36 against zone (2018+); short and mid passes barely change. Offenses throw deep a bit more against both (21.3% vs 19.7% and 22.3% vs 18.7%). Blitzes also sack more (8.2% vs 5.6% of dropbacks), which thrown-pass EPA leaves out, so the blitz is if anything better for the defense than shown.

**Caveats**
- **Observational.** Defenses pick their look pre-snap in response to the offense's formation, which we do not control for; adjustment covers down, distance, field position, score and time only.
- **EPA does not value the clock.** Running with a fourth-quarter lead can be right even with negative EPA; this analysis cannot say it is wrong.
- **Calls exclude sacks and scrambles** (results of a dropback, not calls), and man/zone looks exist from 2018 only.
- **The interaction test has limited power** for subtle effects: "no detectable dependence" is not "no dependence."
